# 03 - Interpretación Avanzada: Cargas Factoriales, Círculo de Correlaciones y Biplots

## Curso de Machine Learning: Modelos No Supervisados

Proyectar un conjunto de datos en un plano 2D formado por PC1 y PC2 nos permite observar agrupamientos y tendencias globales. Sin embargo, para que este análisis sea riguroso y accionable para expertos de dominio o comités de negocio, debemos responder dos preguntas cruciales:
1. **¿Qué significa físicamente o económicamente cada componente?** (¿Qué variables empujan la PC1 hacia la derecha y cuáles hacia la izquierda?)
2. **¿Cómo interactúan las variables entre sí?** (¿Cuáles están fuertemente correlacionadas positivamente, cuáles son ortogonales y cuáles se oponen?)

En este cuaderno exploraremos la escuela francesa de Análisis de Datos Multivariados (Benzécri, Lebart) a través de tres herramientas de interpretación profunda:
- **Cargas Factoriales (*Factor Loadings*)** y su distinción formal frente a los autovectores simples (`pca.components_`).
- **Círculo de Correlaciones (*Correlation Circle*)** y Calidad de Representación ($\text{Cos}^2$).
- **El Biplot de Gabriel (1971)**: Proyección simultánea y coherente de observaciones (filas) y variables (columnas) en el plano factorial.

---

### Contenidos
1. Autovectores (`components_`) vs. Cargas Factoriales de Correlación (*Loadings*)
2. Formulación Matemática de las Cargas: $L_{jk} = v_{jk} \sqrt{\lambda_k} = \text{Corr}(x_j, PC_k)$
3. Calidad de Representación de una Variable en el Plano: $\text{Cos}^2$
4. El Círculo de Correlaciones Unitario y Geometría de Ángulos
5. El Biplot de Gabriel: Interpretación Cruzada de Muestras y Variables
6. Contribución Relativa de una Variable a una Componente ($Contrib$)
7. Caso Práctico Completo sobre el Dataset Wine
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Autovectores vs. Cargas Factoriales (*Factor Loadings*)

Existe una confusión recurrente entre los practicantes de Scikit-Learn respecto al atributo `pca.components_`:
- `pca.components_` contiene los **autovectores unitarios $\mathbf{V}^T$** (filas con norma euclidiana exactamente igual a 1: $\|\mathbf{v}_k\|_2 = 1$). Indican únicamente la dirección geométrica de los ejes.
- En estadística y psicometría, las **Cargas Factoriales (*Loadings*)** corresponden a la **correlación lineal de Pearson entre la variable original estandarizada $x_j$ y la componente principal $PC_k$**:

$$L_{jk} = \text{Corr}(x_j, PC_k) = v_{jk} \cdot \sqrt{\lambda_k}$$

Donde $\lambda_k$ es el autovalor (varianza) de la componente $k$.

### Interpretación de las Cargas ($L_{jk} \in [-1, +1]$):
- $L_{jk} \approx +1$: La variable $x_j$ tiene una correlación positiva casi perfecta con la componente $k$.
- $L_{jk} \approx -1$: La variable $x_j$ tiene una correlación negativa extrema con la componente $k$.
- $L_{jk} \approx 0$: La variable $x_j$ es ortogonal e independiente de la componente $k$.


In [ ]:
# Carga de datos y cálculo explícito de cargas factoriales
wine = load_wine(as_frame=True)
X = wine.data
feature_names = X.columns.tolist()

X_sc = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(X_sc)

# 1. Autovectores V^T (norma 1)
autovectores_V = pca.components_.T

# 2. Cargas factoriales (Loadings = V * sqrt(lambdas))
lambdas = pca.explained_variance_
cargas_loadings = autovectores_V * np.sqrt(lambdas)

df_cargas = pd.DataFrame(cargas_loadings, index=feature_names, columns=['Carga_PC1', 'Carga_PC2'])
print("Cargas Factoriales (Correlación variable-componente):")
display(df_cargas.round(3))


## 2. Calidad de Representación ($\text{Cos}^2$) y Círculo de Correlaciones

### Calidad de Representación ($\text{Cos}^2$)
Para cada variable $x_j$, la suma de los cuadrados de sus cargas en el plano factorial 2D define qué tan bien representada está en ese plano:
$$\text{Cos}^2(x_j, \text{Plano}_{1,2}) = L_{j, 1}^2 + L_{j, 2}^2 \in [0, 1]$$
- Si $\text{Cos}^2 \approx 1$: La flecha de la variable toca el borde del círculo unitario. El plano 2D captura casi toda la variabilidad de esa variable.
- Si $\text{Cos}^2 \approx 0$: La flecha queda muy corta cerca del origen. Sus variaciones ocurren en componentes superiores ($PC_3, PC_4, \dots$) y no debe interpretarse en este plano.

### Interpretación Angular entre Flechas en el Círculo:
- **Ángulo agudo ($\|\theta\| \approx 0^\circ$)**: Las dos variables tienen **fuerte correlación positiva**.
- **Ángulo recto ($\theta \approx 90^\circ$)**: Las dos variables son **ortogonales e incorrelacionadas**.
- **Ángulo obtuso / opuesto ($\|\theta\| \approx 180^\circ$)**: Las dos variables tienen **fuerte correlación negativa**.


In [ ]:
# Gráfico del Círculo de Correlaciones
cos2 = np.sum(cargas_loadings ** 2, axis=1)

fig, ax = plt.subplots(figsize=(9, 9))
# Dibujar circulo unitario
circulo = plt.Circle((0, 0), 1.0, color='gray', fill=False, linestyle='--', lw=1.5)
circulo_medio = plt.Circle((0, 0), 0.707, color='lightgray', fill=False, linestyle=':', lw=1)
ax.add_artist(circulo)
ax.add_artist(circulo_medio)

# Dibujar flechas para cada variable
for i, var_name in enumerate(feature_names):
    x_val, y_val = cargas_loadings[i, 0], cargas_loadings[i, 1]
    # Color segun calidad de representacion Cos^2
    color_arrow = plt.cm.viridis(cos2[i])
    ax.annotate('', xy=(x_val, y_val), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', lw=2.0, color=color_arrow))
    # Etiqueta con ligero desplazamiento
    ax.text(x_val * 1.08, y_val * 1.08, var_name, fontsize=10, fontweight='bold',
            ha='center', va='center', color='darkslategray')

ax.axhline(0, color='black', lw=1)
ax.axvline(0, color='black', lw=1)
ax.set_xlim(-1.25, 1.25)
ax.set_ylim(-1.25, 1.25)
ax.set_aspect('equal')
ax.set_title('Círculo de Correlaciones de Pearson (PC1 vs PC2)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. El Biplot de Gabriel: Proyección Simultánea de Muestras y Variables

El **Biplot** (K. R. Gabriel, 1971) superpone en una sola gráfica:
1. **Las observaciones individuales (puntos)**: Proyectadas sobre las componentes principales.
2. **Las variables (vectores/flechas)**: Escaladas proporcionalmente.

### Regla de Oro para Leer un Biplot:
- Si una muestra está situada en el cuadrante hacia donde apunta la flecha de una variable, esa muestra tiene un **valor inusualmente alto** en esa variable.
- Si una muestra está en el cuadrante exactamente opuesto al sentido de la flecha, tiene un **valor inusualmente bajo** en esa variable.


In [ ]:
# Construcción del Biplot Profesional
X_proyectado = pca.transform(X_sc)
y_cepas = wine.target
nombres_cepas = wine.target_names

fig, ax = plt.subplots(figsize=(12, 7.5))

# 1. Graficar muestras por cepa
paleta = ['#1f77b4', '#2ca02c', '#d62728']
for c_idx, c_nombre in enumerate(nombres_cepas):
    mask = (y_cepas == c_idx)
    ax.scatter(X_proyectado[mask, 0], X_proyectado[mask, 1], c=paleta[c_idx],
               label=c_nombre, s=50, alpha=0.75, edgecolors='k')

# 2. Superponer flechas de variables escaladas por factor visual (3.5)
escala_flechas = 3.5
for i, var_name in enumerate(feature_names):
    vx, vy = cargas_loadings[i, 0] * escala_flechas, cargas_loadings[i, 1] * escala_flechas
    ax.annotate('', xy=(vx, vy), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', lw=1.6, color='black'))
    ax.text(vx * 1.05, vy * 1.05, var_name, fontsize=9, fontweight='bold', color='navy')

ax.axhline(0, color='gray', linestyle=':', lw=1)
ax.axvline(0, color='gray', linestyle=':', lw=1)
ax.set_title('Biplot de PCA: Observaciones y Vectores de Atributos Químicos', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza) -> Estructura Fenólica y Alcohol', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza) -> Intensidad Cromática vs Tono', fontsize=11, fontweight='bold')
ax.legend(title='Cepa de Vino', loc='upper right', frameon=True)
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Cargas de Correlación (*Loadings*)**: $L_{jk} = v_{jk} \sqrt{\lambda_k}$ miden el grado de correlación lineal directo entre la variable $x_j$ y el factor latente $PC_k$.
2. **Círculo de Correlaciones**: La cercanía al borde unitario indica calidad de representación ($\text{Cos}^2$), y el ángulo entre flechas aproxima el coeficiente de correlación entre pares de variables.
3. **Biplot**: Integra la geografía de las muestras con las fuerzas impulsoras de las variables, facilitando la interpretación diagnóstica de cualquier dataset multivariado.

En el siguiente cuaderno, **04_PCA_en_Pipelines_Aceleracion_y_Mitigacion_Colinealidad.ipynb**, utilizaremos PCA como paso de preprocesamiento para eliminar la multicolinealidad y acelerar modelos supervisados.
